# Accuracy and efficiency of the methods 

In [ ]:
#    APM41012EP course notebook - Chapter 3 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Accuracy and efficiency of quadrature methods - cost / accuracy diagram 
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import plotly.graph_objs as go
from scipy.integrate import newton_cotes
from scipy.special import roots_legendre
import warnings
warnings.filterwarnings('ignore')


The quadrature error of a function $f$ on the interval $[a,b]$, when a composite formula is used in which the interval has been split into $N$ sub-intervals of equal size $h=(b -a)/N$, is written:

$$
err =  \int_a^b f(x) {\mathrm d}x - \sum_{j=0}^{N-1} h\,\sum_{i=0}^{s-1}b_i f(x_j+c_i h).
$$

We are going to evaluate this error as a function of the number of evaluations $fe$ of the function $f$ in the computation of the quadrature ($fe = N(s-1)+1$ for a Newton-Cotes formula). This number is an estimate of the work to be done and is considered to be proportional to the computing time on a computer. 

For $N$, in the experiments that follow, we typically take $N=1,2,4,8, \ldots 2048 = 2^{11}$ in order to estimate the results for a single interval, as well as for a series of uniform subdivisions of the integration interval. We consider two functions with two degrees of nonlinearity, each depending on a frequency that will be taken equal to 2 or 20 in the course of the assessment of the accuracy and the efficiency of the methods. The aim is to represent functions that are more or less well approximated by an interpolating polynomial on the integration interval, depending on the degree of nonlinearity, and that oscillate more or less, so as to reach firm conclusions.

In order to observe the results for two functions and these two frequencies, we  plot the logarithm of the error $\log(err)$ as a function of the logarithm of the number of function evaluations $fe$.  This makes it possible, on the one hand, to check that we do have a method of a given order by observing the affine decay of the logarithm of the error in the logarithmic diagram and, on the other hand, to compare the methods in a cost / accuracy diagram, classical in numerical analysis, which allows one to compare the efficiency of the methods at fixed accuracy or, at fixed cost, to compare the accuracy! 


In [ ]:
def integral_step(f, xmin, xmax, r, w):
    h = xmax-xmin
    return h*np.sum(w * f(xmin+h*r))

def integrate(f, xmin, xmax, n, stage, quadrature="newton"):
    if quadrature == "newton":
        if (stage==17):
            # analytical weights as rational numbers
            w = 0.5*np.array([(15043611773/488462349375), (127626606592/488462349375),  (-(3994025216/10854718875)), (166442371072/97692469875), (-(35081792864/8881133625)), (464176543744/54273594375), (-(6806534407936/488462349375)),  (1873775003648/97692469875), (-(75809177572/3618239625)), (1873775003648/97692469875), (-(6806534407936/488462349375)),  (464176543744/54273594375), ( -(35081792864/8881133625)), (166442371072/97692469875), ( -(3994025216/10854718875)), (127626606592/488462349375), (15043611773/488462349375)])
        else:
            w, _ = newton_cotes(stage-1, equal=1)
            w = w/(stage-1)
        r = np.linspace(0, 1, stage)
    elif quadrature == "gauss":
        r, w = roots_legendre(stage)
        r = 0.5*(r+1)
        w = 0.5*w
    else:
        print(f"quadrature : {quadrature} is unknown")
        exit(1)
                
    x = np.linspace(xmin, xmax, n+1)
    res = 0.
    for i in range(n):
        res += integral_step(f, x[i], x[i+1], r, w)
    return res

## Newton-Cotes formulas 

### Integration of $f(x) = \cos(\omega x)$

In [ ]:
def f(x, omega):
    return np.cos(omega*x)

In [ ]:
# Integration over [0,2]
xmin = 0
xmax = 2

# frequency
omega = 2

x = np.linspace(xmin,xmax,1000)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f(omega, x)))
fig.add_shape(type="line", x0=0, y0=0, x1=2, y1=0)
fig.add_shape(type="line", x0=0, y0=-1.1, x1=0, y1=1.1)
buttons=[dict(label="omega=2",  method="update", args=[{"y": [f(x, 2)]}]),
         dict(label="omega=20", method="update", args=[{"y": [f(x, 20)]}])]
fig.update_layout(title={'text':'f(x) = cos(\u03C9 x)', 'x':0.5}, updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x = 0.2, y = 1.2)])
fig.show()

In [ ]:
n = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048])

err_trap   = np.zeros((2, n.size))
err_newton = np.zeros((2, n.size))
err_boole  = np.zeros((2, n.size))
err_weddle = np.zeros((2, n.size))
err_haut10 = np.zeros((2, n.size))
err_haut17 = np.zeros((2, n.size))

for i, omega_i in enumerate([2, 20]):
    res_exa = (1/omega_i)*np.sin(omega_i*xmax) - (1/omega_i)*np.sin(omega_i*xmin)
    fct = lambda x: f(x, omega_i)

    for j, nj in enumerate(n):
    
        # trapezoid
        res_trap = integrate(fct, xmin, xmax, nj, 2, quadrature="newton")
        err_trap[i,j] = abs(res_exa - res_trap)

        # Newton4
        res_newton = integrate(fct, xmin, xmax, nj, 4, quadrature="newton")
        err_newton[i,j] = abs(res_exa - res_newton)
        
        # Boole
        res_boole = integrate(fct, xmin, xmax, nj, 5, quadrature="newton")
        err_boole[i,j] = abs(res_exa - res_boole)

        # Weddle
        res_weddle = integrate(fct, xmin, xmax, nj, 7, quadrature="newton")
        err_weddle[i,j] = abs(res_exa - res_weddle)
        if (err_weddle[i,j]==0):  err_weddle[i,j]=1e-16

        # haut10
        res_haut10 = integrate(fct, xmin, xmax, nj, 10, quadrature="newton")
        err_haut10[i,j] = abs(res_exa - res_haut10)
        if (err_haut10[i,j]==0):  err_haut10[i,j]=1e-16

        #haut17
        res_haut17 = integrate(fct, xmin, xmax, nj, 17, quadrature="newton")
        err_haut17[i,j] = abs(res_exa - res_haut17)
        if (err_haut17[i,j]==0):  err_haut17[i,j]=1e-16

fig = go.Figure()
fig.add_trace(go.Scatter(x=2*n, y=err_trap[0], name="Trapezoid (2 points, order 1)"))
fig.add_trace(go.Scatter(x=4*n, y=err_newton[0], name="Newton (4 points, order 3)"))
fig.add_trace(go.Scatter(x=5*n, y=err_boole[0], name="Boole (5 points, order 5)"))
fig.add_trace(go.Scatter(x=7*n, y=err_weddle[0], name="Weddle (7 points, order 7)"))
fig.add_trace(go.Scatter(x=10*n, y=err_haut10[0], name="(10 points, order 9)"))
fig.add_trace(go.Scatter(x=17*n, y=err_haut17[0], name="(17 points, order 17)"))

buttons=[dict(label="omega=2",  method="update", args=[{"y": [err_trap[0], err_newton[0], err_boole[0], err_weddle[0], err_haut10[0], err_haut17[0]]}]),
         dict(label="omega=20", method="update", args=[{"y": [err_trap[1], err_newton[1], err_boole[1], err_weddle[1], err_haut10[1], err_haut17[1]]}])]

fig.update_xaxes(type="log", exponentformat = 'e', title="Number of evaluations of the function f(x)")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")

fig.update_layout(updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x = 0.2, y = 1.15)],
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))

fig.show()

### Integration of $g(x) = \cos(\omega x) \exp(\sin(\omega x))$

In [ ]:
def g(x, omega):
    return np.cos(omega*x)*np.exp(np.sin(omega*x))

In [ ]:
# Integration over [0,2]
xmin = 0
xmax = 2

# frequency
omega = 2

x = np.linspace(xmin,xmax,1000)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f(omega, x)))
fig.add_shape(type="line", x0=0, y0=0, x1=2, y1=0)
fig.add_shape(type="line", x0=0, y0=-1.5, x1=0, y1=1.5)
buttons=[dict(label="omega=2",  method="update", args=[{"y": [g(x, 2)]}]),
         dict(label="omega=20", method="update", args=[{"y": [g(x, 20)]}])]
fig.update_layout(title={'text':'g(x) = cos(omega x) exp(sin(omega x))', 'x':0.5}, updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x = 0.2, y = 1.2)])
fig.show()

In [ ]:
n = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048])

err_trap   = np.zeros((2, n.size))
err_newton = np.zeros((2, n.size))
err_boole  = np.zeros((2, n.size))
err_weddle = np.zeros((2, n.size))
err_haut10 = np.zeros((2, n.size))
err_haut17 = np.zeros((2, n.size))

for i, omega_i in enumerate([2, 20]):
    res_exa = (1/omega_i)*np.exp(np.sin(omega_i*xmax)) - (1/omega_i)*np.exp(np.sin(omega_i*xmin))
    fct = lambda x: g(x, omega_i)

    for j, nj in enumerate(n):
    
        # trapezoid
        res_trap = integrate(fct, xmin, xmax, nj, 2, quadrature="newton")
        err_trap[i,j] = abs(res_exa - res_trap)

        # Newton4
        res_newton = integrate(fct, xmin, xmax, nj, 4, quadrature="newton")
        err_newton[i,j] = abs(res_exa - res_newton)
        
        # Boole
        res_boole = integrate(fct, xmin, xmax, nj, 5, quadrature="newton")
        err_boole[i,j] = abs(res_exa - res_boole)

        # Weddle
        res_weddle = integrate(fct, xmin, xmax, nj, 7, quadrature="newton")
        err_weddle[i,j] = abs(res_exa - res_weddle)
        if (err_weddle[i,j]==0):  err_weddle[i,j]=1e-16

        # haut10
        res_haut10 = integrate(fct, xmin, xmax, nj, 10, quadrature="newton")
        err_haut10[i,j] = abs(res_exa - res_haut10)
        if (err_haut10[i,j]==0):  err_haut10[i,j]=1e-16

        #haut17
        res_haut17 = integrate(fct, xmin, xmax, nj, 17, quadrature="newton")
        err_haut17[i,j] = abs(res_exa - res_haut17)
        if (err_haut17[i,j]==0):  err_haut17[i,j]=1e-16

fig = go.Figure()
fig.add_trace(go.Scatter(x=2*n, y=err_trap[0], name="Trapezoid (2 points, order 1)"))
fig.add_trace(go.Scatter(x=4*n, y=err_newton[0], name="Newton (4 points, order 3)"))
fig.add_trace(go.Scatter(x=5*n, y=err_boole[0], name="Boole (5 points, order 5)"))
fig.add_trace(go.Scatter(x=7*n, y=err_weddle[0], name="Weddle (7 points, order 7)"))
fig.add_trace(go.Scatter(x=10*n, y=err_haut10[0], name="(10 points, order 9)"))
fig.add_trace(go.Scatter(x=17*n, y=err_haut17[0], name="(17 points, order 17)"))

buttons=[dict(label="omega=2",  method="update", args=[{"y": [err_trap[0], err_newton[0], err_boole[0], err_weddle[0], err_haut10[0], err_haut17[0]]}]),
         dict(label="omega=20", method="update", args=[{"y": [err_trap[1], err_newton[1], err_boole[1], err_weddle[1], err_haut10[1], err_haut17[1]]}])]

fig.update_xaxes(type="log", exponentformat = 'e', title="Number of evaluations of the function g(x)")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")

fig.update_layout(updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x = 0.2, y = 1.15)],
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))

fig.show()

Several conclusions can be drawn from these graphs: 
- whereas in the case of the simple cosine function the first impression is that the most efficient approach is to increase the order of the quadrature formula on a single interval, as soon as one increases the frequency or strengthens the nonlinearity (or even both!), the use of composite formulas turns out to be very efficient, without having to deal with the conditioning and stability issues that arise at too high an order, as seen previously.
- increasing the order within composite formulas is efficient, all the more so when an accurate evaluation of the integral is sought.
- in an intermediate regime, several choices of the number of intervals and of the order make it possible to reach the best efficiency. This will have to be compared with the other types of quadrature (Clenshaw-Curtis and Gauss-Legendre).


## Gauss quadrature formulas                 

### Integration of $f(x) = \cos(\omega x)$  

In [ ]:
n = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048])

err_trap   = np.zeros((2, n.size))
err_newton = np.zeros((2, n.size))
err_boole  = np.zeros((2, n.size))
err_weddle = np.zeros((2, n.size))
err_haut17 = np.zeros((2, n.size))
err_gauss_1 = np.zeros((2, n.size))
err_gauss_2 = np.zeros((2, n.size))
err_gauss_3 = np.zeros((2, n.size))
err_gauss_4 = np.zeros((2, n.size))
err_gauss_9 = np.zeros((2, n.size))

for i, omega_i in enumerate([2, 20]):
    res_exa = (1/omega_i)*np.sin(omega_i*xmax) - (1/omega_i)*np.sin(omega_i*xmin)
    fct = lambda x: f(x, omega_i)

    for j, nj in enumerate(n):
        
        # trapezoid
        res_trap = integrate(fct, xmin, xmax, nj, 2, quadrature="newton")
        err_trap[i,j] = abs(res_exa - res_trap)
        # Gauss 1
        res_gauss_1 = integrate(fct, xmin, xmax, nj, 1, quadrature="gauss")
        err_gauss_1[i,j] = abs(res_exa - res_gauss_1)

        # Newton4
        res_newton = integrate(fct, xmin, xmax, nj, 4, quadrature="newton")
        err_newton[i,j] = abs(res_exa - res_newton)
        # Gauss 2
        res_gauss_2 = integrate(fct, xmin, xmax, nj, 2, quadrature="gauss")
        err_gauss_2[i,j] = abs(res_exa - res_gauss_2)
        
        # Boole
        res_boole = integrate(fct, xmin, xmax, nj, 5, quadrature="newton")
        err_boole[i,j] = abs(res_exa - res_boole)
        # Gauss 3
        res_gauss_3 = integrate(fct, xmin, xmax, nj, 3, quadrature="gauss")
        err_gauss_3[i,j] = abs(res_exa - res_boole)
        if (err_gauss_3[i,j]==0): err_gauss_3[i,j]=1e-16
            
         # Weddle
        res_weddle = integrate(fct, xmin, xmax, nj, 7, quadrature="newton")
        err_weddle[i,j] = abs(res_exa - res_weddle)
        if (err_weddle[i,j]==0):  err_weddle[i,j]=1e-16
        # Gauss 4
        res_gauss_4 = integrate(fct, xmin, xmax, nj, 4, quadrature="gauss")
        err_gauss_4[i,j] = abs(res_exa - res_weddle)
        if (err_gauss_4[i,j]==0):  err_gauss_4[i,j]=1e-16

        #haut17
        res_haut17 = integrate(fct, xmin, xmax, nj, 17, quadrature="newton")
        err_haut17[i,j] = abs(res_exa - res_haut17)
        if (err_haut17[i,j]==0):  err_haut17[i,j]=1e-16
        # Gauss 9
        res_gauss_9 = integrate(fct, xmin, xmax, nj, 9, quadrature="gauss")
        err_gauss_9[i,j] = abs(res_exa - res_gauss_9)
        if (err_gauss_9[i,j]==0):  err_gauss_9[i,j]=1e-16


fig = go.Figure()
fig.add_trace(go.Scatter(x=n, y=err_gauss_1[0], name="Gauss (1 point, order 1)", line_color="#636EFA"))
fig.add_trace(go.Scatter(x=2*n, y=err_trap[0], name="Trapezoid (2 points, order 2)", line_color="#636EFA", line_dash="dash"))

fig.add_trace(go.Scatter(x=2*n, y=err_gauss_2[0], name="Gauss (2 points, order 3)", line_color="#EF553B"))
fig.add_trace(go.Scatter(x=4*n, y=err_newton[0], name="Newton (4 points, order 3)", line_color="#EF553B", line_dash="dash"))

fig.add_trace(go.Scatter(x=3*n, y=err_gauss_3[0], name="Gauss (3 points, order 5)", line_color="#00CC96"))
fig.add_trace(go.Scatter(x=5*n, y=err_boole[0], name="Boole (5 points, order 5)", line_color="#00CC96", line_dash="dash"))

fig.add_trace(go.Scatter(x=4*n, y=err_gauss_4[0], name="Gauss (4 points, order 7)", line_color="#AB63FA"))
fig.add_trace(go.Scatter(x=7*n, y=err_weddle[0], name="Weddle (7 points, order 7)", line_color="#AB63FA", line_dash="dash"))

fig.add_trace(go.Scatter(x=10*n, y=err_gauss_9[0], name="Gauss (9 points, order 17)", line_color="#FFA15A"))
fig.add_trace(go.Scatter(x=17*n, y=err_haut17[0], name="(17 points, order 17)", line_color="#FFA15A", line_dash="dash"))


buttons=[dict(label="omega=2",  method="update", args=[{"y": [err_gauss_1[0], err_trap[0], err_gauss_2[0], err_newton[0],  err_gauss_3[0], err_boole[0], err_gauss_4[0], err_weddle[0], err_gauss_9[0], err_haut17[0]]}]),
         dict(label="omega=20", method="update", args=[{"y": [err_gauss_1[1], err_trap[1], err_gauss_2[1], err_newton[1],  err_gauss_3[1], err_boole[1], err_gauss_4[1], err_weddle[1], err_gauss_9[1], err_haut17[1]]}])]

fig.update_xaxes(type="log", exponentformat = 'e', title="Number of evaluations of the function f(x)")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")

fig.update_layout(updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x = 0.2, y = 1.20)],
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))

fig.show()

### Integration of $g(x) = \cos(\omega x) \exp(\sin(\omega x)$ 

In [ ]:
n = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048])

err_trap   = np.zeros((2, n.size))
err_newton = np.zeros((2, n.size))
err_boole  = np.zeros((2, n.size))
err_weddle = np.zeros((2, n.size))
err_haut17 = np.zeros((2, n.size))
err_gauss_1 = np.zeros((2, n.size))
err_gauss_2 = np.zeros((2, n.size))
err_gauss_3 = np.zeros((2, n.size))
err_gauss_4 = np.zeros((2, n.size))
err_gauss_9 = np.zeros((2, n.size))

for i, omega_i in enumerate([2, 20]):
    res_exa = (1/omega_i)*np.exp(np.sin(omega_i*xmax)) - (1/omega_i)*np.exp(np.sin(omega_i*xmin))
    fct = lambda x: g(x, omega_i)

    for j, nj in enumerate(n):
        
        # trapezoid
        res_trap = integrate(fct, xmin, xmax, nj, 2, quadrature="newton")
        err_trap[i,j] = abs(res_exa - res_trap)
        # Gauss 1
        res_gauss_1 = integrate(fct, xmin, xmax, nj, 1, quadrature="gauss")
        err_gauss_1[i,j] = abs(res_exa - res_gauss_1)

        # Newton4
        res_newton = integrate(fct, xmin, xmax, nj, 4, quadrature="newton")
        err_newton[i,j] = abs(res_exa - res_newton)
        # Gauss 2
        res_gauss_2 = integrate(fct, xmin, xmax, nj, 2, quadrature="gauss")
        err_gauss_2[i,j] = abs(res_exa - res_gauss_2)
        
        # Boole
        res_boole = integrate(fct, xmin, xmax, nj, 5, quadrature="newton")
        err_boole[i,j] = abs(res_exa - res_boole)
        # Gauss 3
        res_gauss_3 = integrate(fct, xmin, xmax, nj, 3, quadrature="gauss")
        err_gauss_3[i,j] = abs(res_exa - res_boole)
        if (err_gauss_3[i,j]==0): err_gauss_3[i,j]=1e-16
            
         # Weddle
        res_weddle = integrate(fct, xmin, xmax, nj, 7, quadrature="newton")
        err_weddle[i,j] = abs(res_exa - res_weddle)
        if (err_weddle[i,j]==0):  err_weddle[i,j]=1e-16
        # Gauss 4
        res_gauss_4 = integrate(fct, xmin, xmax, nj, 4, quadrature="gauss")
        err_gauss_4[i,j] = abs(res_exa - res_weddle)
        if (err_gauss_4[i,j]==0):  err_gauss_4[i,j]=1e-16

        #haut17
        res_haut17 = integrate(fct, xmin, xmax, nj, 17, quadrature="newton")
        err_haut17[i,j] = abs(res_exa - res_haut17)
        if (err_haut17[i,j]==0):  err_haut17[i,j]=1e-16
        # Gauss 9
        res_gauss_9 = integrate(fct, xmin, xmax, nj, 9, quadrature="gauss")
        err_gauss_9[i,j] = abs(res_exa - res_gauss_9)
        if (err_gauss_9[i,j]==0):  err_gauss_9[i,j]=1e-16


fig = go.Figure()
fig.add_trace(go.Scatter(x=n, y=err_gauss_1[0], name="Gauss (1 point, order 1)", line_color="#636EFA"))
fig.add_trace(go.Scatter(x=2*n, y=err_trap[0], name="Trapezoid (2 points, order 1)", line_color="#636EFA", line_dash="dash"))

fig.add_trace(go.Scatter(x=2*n, y=err_gauss_2[0], name="Gauss (2 points, order 3)", line_color="#EF553B"))
fig.add_trace(go.Scatter(x=4*n, y=err_newton[0], name="Newton (4 points, order 3)", line_color="#EF553B", line_dash="dash"))

fig.add_trace(go.Scatter(x=3*n, y=err_gauss_3[0], name="Gauss (3 points, order 5)", line_color="#00CC96"))
fig.add_trace(go.Scatter(x=5*n, y=err_boole[0], name="Boole (5 points, order 5)", line_color="#00CC96", line_dash="dash"))

fig.add_trace(go.Scatter(x=4*n, y=err_gauss_4[0], name="Gauss (4 points, order 7)", line_color="#AB63FA"))
fig.add_trace(go.Scatter(x=7*n, y=err_weddle[0], name="Weddle (7 points, order 7)", line_color="#AB63FA", line_dash="dash"))

fig.add_trace(go.Scatter(x=10*n, y=err_gauss_9[0], name="Gauss (9 points, order 17)", line_color="#FFA15A"))
fig.add_trace(go.Scatter(x=17*n, y=err_haut17[0], name="(17 points, order 17)", line_color="#FFA15A", line_dash="dash"))


buttons=[dict(label="omega=2",  method="update", args=[{"y": [err_gauss_1[0], err_trap[0], err_gauss_2[0], err_newton[0],  err_gauss_3[0], err_boole[0], err_gauss_4[0], err_weddle[0], err_gauss_9[0], err_haut17[0]]}]),
         dict(label="omega=20", method="update", args=[{"y": [err_gauss_1[1], err_trap[1], err_gauss_2[1], err_newton[1],  err_gauss_3[1], err_boole[1], err_gauss_4[1], err_weddle[1], err_gauss_9[1], err_haut17[1]]}])]

fig.update_xaxes(type="log", exponentformat = 'e', title="Number of evaluations of the function g(x)")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")

fig.update_layout(updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x = 0.2, y = 1.20)],
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))

fig.show()

Several conclusions can be drawn from these graphs: 
- Gauss quadrature is clearly more efficient than a Newton-Cotes quadrature at a fixed order.
- increasing the order within composite formulas is very efficient, and this also holds in the intermediate- and high-accuracy regimes. 
- A strategy clearly emerges: a high-order method of Gauss-Legendre type, coupled with a composition method with a reasonable number of intervals, is the most efficient method at fixed accuracy, whether intermediate or high. Stated differently, at a fixed cost within a reasonable range, this strategy gives the most accurate evaluation of the integral.
